In [1]:
import ee
import geemap
import time
import os
from google.colab import files
import geopandas as gpd
import json
import numpy as np
from glob import glob
from google.colab import drive
import matplotlib.pyplot as plt

# Authenticate and initialize
ee.Authenticate()
ee.Initialize(project='gee-tutorial-470209')

In [2]:
# Define the desired directory path
target_directory = '/content/Aoi'
uploaded = files.upload(target_directory)

Saving AOI_Peruvian_River.geojson to /content/Aoi/AOI_Peruvian_River.geojson


In [3]:
# Load GeoJSON
with open("/content/Aoi/AOI_Peruvian_River.geojson") as f:
    geojson_data = json.load(f)

# Convert to EE Geometry or FeatureCollection
roi = ee.FeatureCollection(geojson_data)

# GPT sugested fucode


In [5]:
from tqdm import tqdm
# =================================================
# SCALING FUNCTIONS
# =================================================

def scale_landsat5(img):
    sr = img.select(['SR_B1','SR_B2','SR_B3','SR_B4','SR_B5','SR_B7'])
    scaled = sr.multiply(0.0000275).add(-0.2)
    return img.addBands(scaled, overwrite=True)

def scale_landsat7(img):
    sr = img.select(['SR_B1','SR_B2','SR_B3','SR_B4','SR_B5','SR_B7'])
    scaled = sr.multiply(0.0000275).add(-0.2)
    return img.addBands(scaled, overwrite=True)

def scale_landsat8(img):
    sr = img.select(['SR_B2','SR_B3','SR_B4','SR_B5','SR_B6','SR_B7'])
    scaled = sr.multiply(0.0000275).add(-0.2)
    return img.addBands(scaled, overwrite=True)

# =================================================
# ACTIVE CHANNEL FUNCTION (CORRECTED)
# =================================================

def get_annual_active_channel(year, satellite):

    start = f"{year-1}-12-01"
    end   = f"{year}-04-01"

    # -------------------------
    # Dataset selection
    # -------------------------
    if satellite == 'LS5':
        collection = "LANDSAT/LT05/C02/T1_L2"
        scale_func = scale_landsat5
        green = 'SR_B2'
        red   = 'SR_B3'
        nir   = 'SR_B4'
        swir1 = 'SR_B5'

    elif satellite == 'LS7':
        collection = "LANDSAT/LE07/C02/T1_L2"
        scale_func = scale_landsat7
        green = 'SR_B2'
        red   = 'SR_B3'
        nir   = 'SR_B4'
        swir1 = 'SR_B5'

    elif satellite == 'LS8':
        collection = "LANDSAT/LC08/C02/T1_L2"
        scale_func = scale_landsat8
        green = 'SR_B3'
        red   = 'SR_B4'
        nir   = 'SR_B5'
        swir1 = 'SR_B6'

    else:
        return None

    col = (ee.ImageCollection(collection)
           .filterBounds(roi)
           .filterDate(start, end)
           .filter(ee.Filter.lt('CLOUD_COVER', 20))
           .map(scale_func))

    count = col.size().getInfo()
    if count == 0:
        return None

    median = col.median().clip(roi)

    # -------------------------
    # Spectral indices
    # -------------------------
    mndwi = median.normalizedDifference([green, swir1])
    ndvi  = median.normalizedDifference([nir, red])

    # -------------------------
    # Binary classification
    # -------------------------
    active = (
        mndwi.gte(-0.4)
        .And(ndvi.lte(0.2))
        .rename('active_channel')
        .uint8()
    )

    # -------------------------
    # Small object removal
    # remove patches < 100 pixels
    # -------------------------
    connected = active.connectedPixelCount(1000, True)

    active_clean = active.updateMask(
        connected.gte(1000)
    )

    # convert back to full binary raster
    active_clean = active_clean.unmask(0).uint8()

    # -------------------------
    # Area calculation
    # -------------------------
    area_img = active_clean.multiply(ee.Image.pixelArea())

    area = area_img.reduceRegion(
        reducer=ee.Reducer.sum(),
        geometry=roi,
        scale=30,
        maxPixels=1e13,
        bestEffort=True
    )

    area_km2 = ee.Number(area.get('active_channel')).divide(1e6)

    # Format Satellite name file name (LS_05, LS_07, LS_08)
    if satellite == 'LS5':
      source_display = 'LS_05'
    elif satellite == 'LS7':
      source_display = 'LS_07'
    elif satellite == 'LS8':
      source_display = 'LS_08'

    # -------------------------
    # Metadata
    # -------------------------
    active_clean = active_clean.set({
        'year': year,
        'satellite': satellite,
        'source_display': source_display,
        'n_images': count,
        'active_area_km2': area_km2,
        'method': 'Boothroyd 2020 binary'
    })
    return active_clean

# =================================================
# YEAR LIST
# =================================================

ls5_years = list(range(1984, 2012))
ls7_years = [2002, 2003, 2012, 2013]
ls8_years = list(range(2014, 2027))

pairs = []
pairs.extend([(y,'LS5') for y in ls5_years])
pairs.extend([(y,'LS7') for y in ls7_years])
pairs.extend([(y,'LS8') for y in ls8_years])

pairs.sort(key=lambda x: x[0])

# =================================================
# PROCESS LOOP
# =================================================

all_masks = []

print("\nGenerating binary masks...\n")

for year, sat in tqdm(pairs):
    try:
        mask = get_annual_active_channel(year, sat)
        if mask:
            all_masks.append(mask)
            try:
                area = mask.get('active_area_km2').getInfo()
                print(f"{sat} {year}  area = {area:.2f} km²")
            except:
                pass
        else:
            print(f"{sat} {year}  no images")

    except Exception as e:
        print(year, sat, str(e))
    time.sleep(1)

print("\nTotal masks created:", len(all_masks))


Generating binary masks...



  0%|          | 0/45 [00:00<?, ?it/s]

LS5 1984  area = 23.66 km²


  2%|▏         | 1/45 [00:02<01:37,  2.21s/it]

LS5 1985  area = 23.80 km²


...

LS8 2025  area = 21.91 km²


 98%|█████████▊| 44/45 [01:23<00:02,  2.36s/it]

LS8 2026  no images


100%|██████████| 45/45 [01:24<00:00,  1.88s/it]


Total masks created: 34


# Export to Drive

In [6]:
# ============================================
# EXPORT ALL MASKS
# ============================================
print("\n🚀 STARTING EXPORTS TO GOOGLE DRIVE")
print("=" * 70)

folder_name = "Peruvian River"
print(f"📁 Export folder: {folder_name}")
print()

export_tasks = []
for img in tqdm(all_masks, desc="Starting exports", unit="file"):
    year = img.get('year').getInfo()
    source_display = img.get('source_display').getInfo()  # This gives "LS_05", "LS_07", or "LS_08"
    # Create filename exactly as requested: FID_08_Year_LS_05_Active_Channel_Mask
    filename = f"Peruvian_River_{year}_{source_display}"
    # Get area for display
    try:
        area = img.get('active_area_km2').getInfo()
        area_str = f" ({area:.2f} km²)"
    except:
        area_str = ""
    task = ee.batch.Export.image.toDrive(
        image=img,
        description=filename,
        folder=folder_name,
        fileNamePrefix=filename,
        region=roi.geometry().bounds().getInfo()["coordinates"],
        scale=30,
        crs='EPSG:32645',
        maxPixels=1e13,
        fileFormat='GeoTIFF'
    )
    task.start()
    export_tasks.append({'task': task, 'year': year, 'source': source_display, 'filename': filename})
    tqdm.write(f"  🚀 {filename}: Export started{area_str}")
    time.sleep(3)
print(f"\n✅ Started {len(export_tasks)} export tasks to folder: {folder_name}")
print()


🚀 STARTING EXPORTS TO GOOGLE DRIVE
📁 Export folder: Peruvian River



Starting exports:   0%|          | 0/34 [00:00<?, ?file/s]

  🚀 Peruvian_River_1984_LS_05: Export started (23.66 km²)


Starting exports:   3%|▎         | 1/34 [00:04<02:03,  3.73s/file]

  🚀 Peruvian_River_1985_LS_05: Export started (23.80 km²)


...

  🚀 Peruvian_River_2024_LS_08: Export started (21.89 km²)


Starting exports:  97%|█████████▋| 33/34 [01:57<00:03,  3.47s/file]

  🚀 Peruvian_River_2025_LS_08: Export started (21.91 km²)


Starting exports: 100%|██████████| 34/34 [02:00<00:00,  3.55s/file]


✅ Started 34 export tasks to folder: Peruvian River

